In [1]:
import sys
import pandas as pd

print("Python version:", sys.version)
print("Pandas version:", pd.__version__)

Python version: 3.14.7 (tags/v3.14.7:823f032, Aug  5 2026, 10:51:32) [MSC v.1944 64 bit (AMD64)]
Pandas version: 3.0.6


In [2]:
from pathlib import Path

DATA_PATH = Path("../data/CIC-IDS-2017-V2.csv")

df = pd.read_csv(DATA_PATH)

print("Dataset loaded successfully!")
print("Rows:", len(df))
print("Columns:", len(df.columns))

Dataset loaded successfully!
Rows: 2695162
Columns: 79


# ETIE Anomaly Detection
## 01 - CIC-IDS-2017 Dataset Inspection

In [5]:
for column in df.columns:
    print(column)

Destination_Port
Flow_Duration
Total_Fwd_Packets
Total_Backward_Packets
Total_Length_of_Fwd_Packets
Total_Length_of_Bwd_Packets
Fwd_Packet_Length_Max
Fwd_Packet_Length_Min
Fwd_Packet_Length_Mean
Fwd_Packet_Length_Std
Bwd_Packet_Length_Max
Bwd_Packet_Length_Min
Bwd_Packet_Length_Mean
Bwd_Packet_Length_Std
Flow_Bytes/s
Flow_Packets/s
Flow_IAT_Mean
Flow_IAT_Std
Flow_IAT_Max
Flow_IAT_Min
Fwd_IAT_Total
Fwd_IAT_Mean
Fwd_IAT_Std
Fwd_IAT_Max
Fwd_IAT_Min
Bwd_IAT_Total
Bwd_IAT_Mean
Bwd_IAT_Std
Bwd_IAT_Max
Bwd_IAT_Min
Fwd_PSH_Flags
Bwd_PSH_Flags
Fwd_URG_Flags
Bwd_URG_Flags
Fwd_Header_Length
Bwd_Header_Length
Fwd_Packets/s
Bwd_Packets/s
Min_Packet_Length
Max_Packet_Length
Packet_Length_Mean
Packet_Length_Std
Packet_Length_Variance
FIN_Flag_Count
SYN_Flag_Count
RST_Flag_Count
PSH_Flag_Count
ACK_Flag_Count
URG_Flag_Count
CWE_Flag_Count
ECE_Flag_Count
Down/Up_Ratio
Average_Packet_Size
Avg_Fwd_Segment_Size
Avg_Bwd_Segment_Size
Fwd_Header_Length.1
Fwd_Avg_Bytes/Bulk
Fwd_Avg_Packets/Bulk
Fwd_Avg_Bulk_Ra

In [6]:
print(df.iloc[:, -1].value_counts())

label
BENIGN                        2096484
DoS Hulk                       172849
Comb                           172800
DDoS                           128016
PortScan                        90819
DoS GoldenEye                   10286
FTP-Patator                      5933
DoS slowloris                    5385
DoS Slowhttptest                 5228
SSH-Patator                      3219
Bot                              1953
Web Attack � Brute Force         1470
Web Attack � XSS                  652
Infiltration                       36
Web Attack � Sql Injection         21
Heartbleed                         11
Name: count, dtype: int64


In [7]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 2695162 entries, 0 to 2695161
Data columns (total 79 columns):
 #   Column                       Dtype  
---  ------                       -----  
 0   Destination_Port             float64
 1   Flow_Duration                float64
 2   Total_Fwd_Packets            float64
 3   Total_Backward_Packets       float64
 4   Total_Length_of_Fwd_Packets  float64
 5   Total_Length_of_Bwd_Packets  float64
 6   Fwd_Packet_Length_Max        float64
 7   Fwd_Packet_Length_Min        float64
 8   Fwd_Packet_Length_Mean       float64
 9   Fwd_Packet_Length_Std        float64
 10  Bwd_Packet_Length_Max        float64
 11  Bwd_Packet_Length_Min        float64
 12  Bwd_Packet_Length_Mean       float64
 13  Bwd_Packet_Length_Std        float64
 14  Flow_Bytes/s                 float64
 15  Flow_Packets/s               float64
 16  Flow_IAT_Mean                float64
 17  Flow_IAT_Std                 float64
 18  Flow_IAT_Max                 float64
 19  Flow_IAT_Mi

In [8]:
missing_values = df.isnull().sum()

print(missing_values[missing_values > 0])

Series([], dtype: int64)


In [9]:
import numpy as np

numeric_df = df.select_dtypes(include=np.number)

infinite_values = np.isinf(numeric_df).sum()

print(infinite_values[infinite_values > 0])

Series([], dtype: int64)


In [10]:
print(
    "Fwd_Header_Length and Fwd_Header_Length.1 identical:",
    df["Fwd_Header_Length"].equals(df["Fwd_Header_Length.1"])
)

Fwd_Header_Length and Fwd_Header_Length.1 identical: False


In [13]:
print(df[["Fwd_Header_Length", "Fwd_Header_Length.1"]].head(10))

   Fwd_Header_Length  Fwd_Header_Length.1
0           0.999856             0.999856
1           0.999856             0.999856
2           0.999856             0.999856
3           0.999856             0.999856
4           0.999856             0.999856
5           0.999856             0.999856
6           0.999856             0.999856
7           0.999856             0.999856
8           0.999856             0.999856
9           0.999856             0.999856


In [14]:
pd.set_option("display.max_rows", 100)

feature_summary = df.describe().T[
    ["min", "max", "mean", "std"]
]

print(feature_summary)

                             min           max          mean           std
Destination_Port             0.0  1.000000e+00  1.244935e-01  2.826447e-01
Flow_Duration                0.0  1.000000e+00  1.293185e-01  2.859797e-01
Total_Fwd_Packets            0.0  1.000000e+00  3.950495e-05  3.496101e-03
Total_Backward_Packets       0.0  1.000000e+00  3.707982e-05  3.501502e-03
Total_Length_of_Fwd_Packets  0.0  1.000000e+00  2.595058e-03  1.136847e-02
Total_Length_of_Bwd_Packets  0.0  1.000000e+00  2.589133e-05  3.538483e-03
Fwd_Packet_Length_Max        0.0  1.000000e+00  8.713753e-03  2.956113e-02
Fwd_Packet_Length_Min        0.0  1.000000e+00  7.726442e-03  2.537674e-02
Fwd_Packet_Length_Mean       0.0  1.000000e+00  9.998679e-03  3.194293e-02
Fwd_Packet_Length_Std        0.0  1.000000e+00  1.014974e-02  4.038210e-02
Bwd_Packet_Length_Max        0.0  1.000000e+00  5.540596e-02  1.034441e-01
Bwd_Packet_Length_Min        0.0  1.000000e+00  1.393960e-02  2.395376e-02
Bwd_Packet_Length_Mean   

In [15]:
print("Number of unique labels:", df["label"].nunique())
print("\nLabels:")
for label in df["label"].unique():
    print(repr(label))

Number of unique labels: 16

Labels:
'BENIGN'
'Infiltration'
'Bot'
'PortScan'
'DDoS'
'FTP-Patator'
'SSH-Patator'
'DoS slowloris'
'DoS Slowhttptest'
'DoS Hulk'
'DoS GoldenEye'
'Heartbleed'
'Web Attack � Brute Force'
'Web Attack � XSS'
'Web Attack � Sql Injection'
'Comb'


In [16]:
difference = (
    df["Fwd_Header_Length"] - df["Fwd_Header_Length.1"]
).abs()

print("Maximum absolute difference:", difference.max())
print("Number of rows with a difference:", (difference > 0).sum())

print(
    df[["Fwd_Header_Length", "Fwd_Header_Length.1"]]
    .corr()
)

Maximum absolute difference: 0.5127811398642048
Number of rows with a difference: 172800
                     Fwd_Header_Length  Fwd_Header_Length.1
Fwd_Header_Length             1.000000             0.995768
Fwd_Header_Length.1           0.995768             1.000000


In [17]:
zero_percentage = (df.drop(columns="label") == 0).mean() * 100

zero_summary = (
    zero_percentage
    .sort_values(ascending=False)
    .to_frame("Zero_Percentage")
)

print(zero_summary.to_string())

                             Zero_Percentage
Bwd_Avg_Bulk_Rate                  93.588512
Bwd_Avg_Packets/Bulk               93.588512
Fwd_Avg_Bytes/Bulk                 93.588512
Fwd_Avg_Packets/Bulk               93.588512
Bwd_PSH_Flags                      93.588512
Bwd_URG_Flags                      93.588512
Fwd_Avg_Bulk_Rate                  93.588512
Bwd_Avg_Bytes/Bulk                 93.588512
CWE_Flag_Count                     93.585543
Fwd_URG_Flags                      93.585543
RST_Flag_Count                     93.563059
ECE_Flag_Count                     93.562947
FIN_Flag_Count                     90.579935
SYN_Flag_Count                     89.027190
Fwd_PSH_Flags                      89.027190
Active_Std                         85.962476
Idle_Std                           85.082789
URG_Flag_Count                     84.096058
Active_Min                         72.854136
Active_Max                         72.854136
Active_Mean                        72.854136
Idle_Min  

In [18]:
unique_counts = df.drop(columns="label").nunique()

print(
    unique_counts
    .sort_values()
    .to_frame("Unique_Values")
    .to_string()
)

                             Unique_Values
FIN_Flag_Count                      164306
PSH_Flag_Count                      168897
Bwd_PSH_Flags                       170427
ACK_Flag_Count                      170504
ECE_Flag_Count                      170541
CWE_Flag_Count                      170556
SYN_Flag_Count                      170606
Fwd_URG_Flags                       170610
min_seg_size_forward                170610
Fwd_Avg_Bytes/Bulk                  170630
Fwd_Avg_Packets/Bulk                170639
Bwd_Avg_Packets/Bulk                170650
URG_Flag_Count                      170663
Fwd_PSH_Flags                       170678
Min_Packet_Length                   170686
Fwd_Header_Length                   170744
Fwd_Avg_Bulk_Rate                   170756
Bwd_Avg_Bytes/Bulk                  170826
RST_Flag_Count                      170868
Bwd_Avg_Bulk_Rate                   170874
Fwd_Packet_Length_Min               170895
Bwd_URG_Flags                       170899
Down/Up_Rat

In [19]:
feature_range = pd.DataFrame({
    "Min": df.drop(columns="label").min(),
    "Max": df.drop(columns="label").max()
})

print(feature_range.to_string())

                             Min           Max
Destination_Port             0.0  1.000000e+00
Flow_Duration                0.0  1.000000e+00
Total_Fwd_Packets            0.0  1.000000e+00
Total_Backward_Packets       0.0  1.000000e+00
Total_Length_of_Fwd_Packets  0.0  1.000000e+00
Total_Length_of_Bwd_Packets  0.0  1.000000e+00
Fwd_Packet_Length_Max        0.0  1.000000e+00
Fwd_Packet_Length_Min        0.0  1.000000e+00
Fwd_Packet_Length_Mean       0.0  1.000000e+00
Fwd_Packet_Length_Std        0.0  1.000000e+00
Bwd_Packet_Length_Max        0.0  1.000000e+00
Bwd_Packet_Length_Min        0.0  1.000000e+00
Bwd_Packet_Length_Mean       0.0  1.000000e+00
Bwd_Packet_Length_Std        0.0  1.000000e+00
Flow_Bytes/s                 0.0  1.000000e+00
Flow_Packets/s               0.0  1.000000e+00
Flow_IAT_Mean                0.0  1.000000e+00
Flow_IAT_Std                 0.0  1.000000e+00
Flow_IAT_Max                 0.0  1.000000e+00
Flow_IAT_Min                 0.0  1.000000e+00
Fwd_IAT_Total

In [20]:
print("Comb rows:", (df["label"] == "Comb").sum())

comb_df = df[df["label"] == "Comb"]

print("\nComb feature statistics:")
print(
    comb_df[
        ["Fwd_Header_Length", "Fwd_Header_Length.1"]
    ].describe().T
)

print("\nDifference statistics:")
comb_difference = (
    comb_df["Fwd_Header_Length"]
    - comb_df["Fwd_Header_Length.1"]
).abs()

print(comb_difference.describe())

print("\nNumber of Comb rows where the two features differ:")
print((comb_difference > 0).sum())

Comb rows: 172800

Comb feature statistics:
                        count          mean           std           min  \
Fwd_Header_Length    172800.0  3.850136e-10  7.088024e-10  4.538533e-30   
Fwd_Header_Length.1  172800.0  1.310763e-01  7.756814e-02  1.752792e-02   

                              25%           50%           75%           max  
Fwd_Header_Length    3.167461e-13  5.099925e-11  4.501630e-10  7.380064e-09  
Fwd_Header_Length.1  7.163277e-02  1.005983e-01  1.783705e-01  5.127811e-01  

Difference statistics:
count    172800.000000
mean          0.131076
std           0.077568
min           0.017528
25%           0.071633
50%           0.100598
75%           0.178371
max           0.512781
dtype: float64

Number of Comb rows where the two features differ:
172800


In [21]:
print("\nMean feature values for each label:")

selected_features = [
    "Fwd_Header_Length",
    "Fwd_Header_Length.1",
    "Flow_Duration",
    "Flow_Bytes/s",
    "Flow_Packets/s",
    "Total_Fwd_Packets",
    "Total_Backward_Packets"
]

print(
    df.groupby("label")[selected_features]
      .mean()
      .sort_index()
      .to_string()
)


Mean feature values for each label:
                            Fwd_Header_Length  Fwd_Header_Length.1  Flow_Duration  Flow_Bytes/s  Flow_Packets/s  Total_Fwd_Packets  Total_Backward_Packets
label                                                                                                                                                     
BENIGN                           9.998547e-01             0.999855   1.012664e-01  1.126396e-01    3.422589e-01       4.718414e-05            4.479715e-05
Bot                              9.998558e-01             0.999856   2.944097e-03  1.120467e-01    3.406672e-01       1.004922e-05            1.150803e-05
Comb                             3.850136e-10             0.131076   1.259138e-10  7.224282e-11    1.550420e-09       3.063458e-10            9.252514e-11
DDoS                             9.998558e-01             0.999856   1.413075e-01  1.119471e-01    3.333594e-01       1.580206e-05            1.115382e-05
DoS GoldenEye                    

In [22]:
# Check for completely duplicated rows
duplicate_count = df.duplicated().sum()

print("Completely duplicated rows:", duplicate_count)
print("Percentage duplicated:", duplicate_count / len(df) * 100)

Completely duplicated rows: 33244
Percentage duplicated: 1.2334694537842252


In [23]:
# Check whether identical feature vectors appear with different labels
feature_columns = df.columns.drop("label")

duplicate_features = df.duplicated(
    subset=feature_columns,
    keep=False
)

print("Rows involved in duplicated feature vectors:", duplicate_features.sum())
print(
    "Percentage:",
    duplicate_features.mean() * 100
)

Rows involved in duplicated feature vectors: 65371
Percentage: 2.4254942745556667


In [24]:
# How many distinct feature vectors exist?
unique_feature_vectors = df[feature_columns].drop_duplicates().shape[0]

print("Total rows:", len(df))
print("Unique feature vectors:", unique_feature_vectors)
print(
    "Repeated feature-vector rows:",
    len(df) - unique_feature_vectors
)

Total rows: 2695162
Unique feature vectors: 2661199
Repeated feature-vector rows: 33963


In [25]:
feature_columns = df.columns.drop("label")

# Count distinct labels associated with each identical feature vector
label_counts_per_vector = (
    df.groupby(list(feature_columns), sort=False)["label"]
      .nunique()
)

print("Feature vectors associated with multiple labels:",
      (label_counts_per_vector > 1).sum())

print(
    "Maximum number of labels for one feature vector:",
    label_counts_per_vector.max()
)

Feature vectors associated with multiple labels: 719
Maximum number of labels for one feature vector: 2


In [26]:
# Show examples where exactly the same feature vector has multiple labels

multi_label_vectors = label_counts_per_vector[
    label_counts_per_vector > 1
]

print("Number of multi-label feature vectors:",
      len(multi_label_vectors))

Number of multi-label feature vectors: 719


In [27]:
# Find feature vectors that occur with more than one label

feature_columns = df.columns.drop("label")

multi_label_groups = (
    df.groupby(list(feature_columns), sort=False)["label"]
      .unique()
)

multi_label_groups = multi_label_groups[
    multi_label_groups.apply(len) > 1
]

print("Multi-label feature vectors:", len(multi_label_groups))

print("\nLabel combinations:")
for labels in multi_label_groups.value_counts().items():
    print(labels)

Multi-label feature vectors: 719

Label combinations:
(<StringArray>
['BENIGN', 'DoS Hulk']
Length: 2, dtype: str, 1)
(<StringArray>
['BENIGN', 'DoS Hulk']
Length: 2, dtype: str, 1)
(<StringArray>
['BENIGN', 'DoS Hulk']
Length: 2, dtype: str, 1)
(<StringArray>
['BENIGN', 'DoS Hulk']
Length: 2, dtype: str, 1)
(<StringArray>
['BENIGN', 'DoS Hulk']
Length: 2, dtype: str, 1)
(<StringArray>
['BENIGN', 'DoS Hulk']
Length: 2, dtype: str, 1)
(<StringArray>
['BENIGN', 'DoS Hulk']
Length: 2, dtype: str, 1)
(<StringArray>
['BENIGN', 'PortScan']
Length: 2, dtype: str, 1)
(<StringArray>
['BENIGN', 'PortScan']
Length: 2, dtype: str, 1)
(<StringArray>
['BENIGN', 'PortScan']
Length: 2, dtype: str, 1)
(<StringArray>
['BENIGN', 'PortScan']
Length: 2, dtype: str, 1)
(<StringArray>
['BENIGN', 'PortScan']
Length: 2, dtype: str, 1)
(<StringArray>
['BENIGN', 'PortScan']
Length: 2, dtype: str, 1)
(<StringArray>
['BENIGN', 'PortScan']
Length: 2, dtype: str, 1)
(<StringArray>
['BENIGN', 'PortScan']
Length: 2, d

In [28]:
from collections import Counter

label_combinations = Counter(
    tuple(sorted(labels))
    for labels in multi_label_groups
)

for combination, count in label_combinations.most_common():
    print(count, "->", combination)

564 -> ('BENIGN', 'PortScan')
150 -> ('BENIGN', 'DoS Hulk')
3 -> ('BENIGN', 'DDoS')
2 -> ('BENIGN', 'DoS slowloris')
